🚀 Love that attitude.

And honestly, this project has followed a very realistic progression.

We started with:

ratings.csv
↓
groupby()
↓
average ratings

Then moved through:

Popularity
↓
User CF
↓
Item CF
↓
Matrix Factorization
↓
Content-Based
↓
Hybrid Recommender

Then into:

Model Persistence
↓
Versioning
↓
FastAPI
↓
Health Checks
↓
Validation

Now you're entering the part many tutorials never reach:

Performance
Observability
Deployment
MLOps


---

# 🚀 Day 13 — API Optimization & Caching

🎯 Goal

Make recommendations faster.

Right now every request:

Request
↓
Generate recommendations
↓
Return response

Even if the same user requests again.

Example:

/recommend/1?n=10

called 100 times

↓

Your system recomputes 100 times.

That's wasteful.


---

🧠 What is Caching?

Instead of:

Compute
Compute
Compute
Compute

Do:

Compute once
↓
Store result
↓
Reuse


---

Real-world analogy

Imagine:

Teacher asks:
What is 5 × 5?

Student:

25

Next minute:

What is 5 × 5?

Would you recalculate?

No.

Use cached answer.


---

🚀 Step 1 — Create cache dictionary

Near top of app.py

recommendation_cache = {}


---

Step 2 — Cache key

Different requests:

/recommend/1?n=10

and

/recommend/1?n=5

must be treated separately.

Create key:

cache_key = f"{user_id}_{n}"


---

Step 3 — Check cache first

At start of endpoint:

cache_key = f"{user_id}_{n}"

if cache_key in recommendation_cache:

    return recommendation_cache[
        cache_key
    ]


---

Step 4 — Store results

Before final return:

response = {

    "model_version":
        app.state.model_name,

    "user_id":
        user_id,

    "recommendation_count":
        len(output),

    "recommendations":
        output,

    "response_time":
        round(
            end-start,
            2
        )
}

Store:

recommendation_cache[
    cache_key
] = response

Return:

return response


---

🚀 Step 5 — Show cache hit

Useful for debugging.

Modify cache block:

if cache_key in recommendation_cache:

    print(
        f"Cache hit: {cache_key}"
    )

    return recommendation_cache[
        cache_key
    ]


---

Test

Call:

/recommend/1?n=10

twice.

Expected:

First:

Response time: 0.48 sec

Second:

Cache hit: 1_10

Almost instant.


---

🧠 Why caching matters

Imagine:

10,000 users

and:

Most users refresh homepage

Without caching:

Repeated expensive computation

With caching:

Reuse previous recommendations

Huge speedup.


---

🚀 Step 6 — Add cache statistics endpoint

Create:

@app.get("/cache-stats")

def cache_stats():

    return {

        "cached_entries":
        len(
            recommendation_cache
        )
    }


---

Test

After several requests:

/cache-stats

returns:

{
  "cached_entries": 12
}


---

🧠 Limitation of current cache

Current cache:

Stored in RAM

If server restarts:

Cache lost

Production systems use:

Redis

for distributed caching.

We'll discuss that later conceptually.


---

🚀 Step 7 — API Metrics Endpoint

Add:

request_count = 0

Top of app.


---

Inside endpoint:

global request_count

request_count += 1


---

Create:

@app.get("/metrics")

def metrics():

    return {

        "total_requests":
        request_count,

        "cache_entries":
        len(
            recommendation_cache
        )
    }


---

Example

{
  "total_requests": 45,
  "cache_entries": 12
}


---

🧠 Why metrics matter

Production teams constantly monitor:

Request count
Latency
Error rate
Cache hit rate

This is the beginning of observability.


---

🎯 Homework

1. Test cache

Call:

/recommend/1?n=10

twice.

Verify:

Cache hit appears


---

2. Check cache size

/cache-stats

after multiple users.


---

3. Think about this

Right now:

Cache grows forever

Question:

How do we remove old cache entries?

That leads into:

🚀 Day 14 — Logging, Monitoring & Cache Expiration (TTL)

where we'll make the service much more production-ready.

